# 🏦 Session 4 — Vector Databases Part 2
## BajajBot v0.4 — Dense · Sparse · Hybrid · Eval · Reranking


**What this notebook builds:**
- ✅ ChromaDB dense index (semantic search)
- ✅ FAISS HNSW index (for QPS/latency benchmarking)
- ✅ BM25 sparse retriever (keyword/product code search)
- ✅ LangChain EnsembleRetriever (hybrid = sparse + dense)
- ✅ Retrieval evaluation: Precision@K, Recall@K, MRR@K, NDCG@K
- ✅ BM25 reranking on ANN candidates
- ✅ CrossEncoder reranking
- ✅ QPS + latency benchmarking (p50/p95/p99)

---


## 📦 Step 0 — Install Dependencies

In [ ]:
# Run once — restart kernel after installation
!pip install -q langchain langchain-community langchain-openai \
               chromadb faiss-cpu rank_bm25 \
               sentence-transformers openai \
               numpy matplotlib

## 🗂️ Step 1 — Build Rich BajajBot FAQ Corpus

**Why this step exists:**  
Sessions 1–3 used a tiny demo corpus (5–10 chunks). For Session 4 we need:
- **50 realistic chunks** covering multiple product lines
- **Exact product codes** (for sparse/keyword search demo)
- **Known ground-truth** per query (for evaluation metrics)
- **Near-duplicate chunks** (to show why reranking matters)

This is the same `bajaj_finance_policy_reference.pdf` content from Session 2,  
now structured as LangChain `Document` objects with rich metadata.

In [14]:
from langchain_core.documents import Document

bajaj_docs = [

    # ── PERSONAL LOANS ──────────────────────────────────────────
    Document(
        page_content="Bajaj Finance Personal Loan BFL-PL-2024 offers loan amounts from ₹1 lakh to ₹40 lakhs. "
                     "Interest rates start at 10.99% per annum. Loan tenure ranges from 12 to 84 months. "
                     "Processing fee is 3.93% of the loan amount (maximum ₹3,93,000). No collateral required.",
        metadata={"source": "faq_pl_001", "category": "personal_loan", "product_code": "BFL-PL-2024",
                  "region": "pan_india", "topic": "loan_details"}
    ),
    Document(
        page_content="Bajaj Finance Personal Loan BFL-PL-2024 eligibility criteria: Minimum age 21 years, "
                     "maximum age 80 years at loan maturity. Minimum monthly salary ₹25,000 for salaried employees. "
                     "Minimum 1 year of employment with current employer. CIBIL score of 685 or above required.",
        metadata={"source": "faq_pl_002", "category": "personal_loan", "product_code": "BFL-PL-2024",
                  "region": "pan_india", "topic": "eligibility"}
    ),
    Document(
        page_content="Documents required for Bajaj Finance Personal Loan BFL-PL-2024: Aadhaar card (mandatory), "
                     "PAN card, last 3 months salary slips, last 6 months bank statement, "
                     "Form 16 or ITR for last 2 years, passport-size photograph. "
                     "Self-employed: GST certificate + last 2 years ITR + CA-certified balance sheet.",
        metadata={"source": "faq_pl_003", "category": "personal_loan", "product_code": "BFL-PL-2024",
                  "region": "pan_india", "topic": "documents"}
    ),
    Document(
        page_content="Personal Loan BFL-PL-2024 EMI calculation example: For ₹5,00,000 loan at 12% per annum "
                     "for 36 months, EMI = ₹16,607. Total interest payable = ₹97,852. "
                     "Total repayment amount = ₹5,97,852. Use the EMI calculator on bajajfinserv.in for custom calculations.",
        metadata={"source": "faq_pl_004", "category": "personal_loan", "product_code": "BFL-PL-2024",
                  "region": "pan_india", "topic": "emi_calculation"}
    ),
    Document(
        page_content="Bajaj Finance Personal Loan BFL-PL-2024-MUMBAI: Special Mumbai zone offer. "
                     "Interest rate 10.49% per annum (0.5% lower than standard rate). "
                     "Available for Mumbai, Thane, Navi Mumbai, Pune residents. "
                     "Offer valid till March 31, 2025. Apply at any Mumbai branch or bajajfinserv.in.",
        metadata={"source": "faq_pl_005", "category": "personal_loan", "product_code": "BFL-PL-2024-MUMBAI",
                  "region": "west_india", "topic": "regional_offer"}
    ),
    Document(
        page_content="Personal Loan BFL-PL-2024-DELHI: Delhi NCR special offer at 10.75% per annum. "
                     "Covers Delhi, Gurgaon, Noida, Faridabad, Ghaziabad. "
                     "Maximum loan amount ₹50 lakhs for government employees. "
                     "Processing fee waived for first 500 applicants this quarter.",
        metadata={"source": "faq_pl_006", "category": "personal_loan", "product_code": "BFL-PL-2024-DELHI",
                  "region": "north_india", "topic": "regional_offer"}
    ),
    Document(
        page_content="How to foreclose a Bajaj Finance Personal Loan BFL-PL-2024: Foreclosure allowed after "
                     "6 EMI payments. Foreclosure charge: 4.72% of outstanding principal + GST. "
                     "Part-prepayment allowed twice a year. Part-prepayment charge: 2.36% of prepaid amount + GST. "
                     "No foreclosure allowed in the first 6 months of loan tenure.",
        metadata={"source": "faq_pl_007", "category": "personal_loan", "product_code": "BFL-PL-2024",
                  "region": "pan_india", "topic": "foreclosure"}
    ),
    Document(
        page_content="Personal Loan BFL-PL-2024 late payment charges: If EMI is not paid by due date, "
                     "penal interest of 2% per month on overdue amount is charged. "
                     "After 30 days overdue, a bounce charge of ₹700 + GST per bounce applies. "
                     "Credit score will be negatively impacted after 30 days of non-payment.",
        metadata={"source": "faq_pl_008", "category": "personal_loan", "product_code": "BFL-PL-2024",
                  "region": "pan_india", "topic": "late_payment"}
    ),

    # ── HOME LOANS ───────────────────────────────────────────────
    Document(
        page_content="Bajaj Housing Finance Home Loan BFL-HL-2024: Loan amount up to ₹5 crores. "
                     "Interest rate starting at 8.50% per annum (floating). Tenure up to 30 years. "
                     "LTV ratio: Up to 90% for loans below ₹30 lakhs, 80% for ₹30–75 lakhs, 75% above ₹75 lakhs.",
        metadata={"source": "faq_hl_001", "category": "home_loan", "product_code": "BFL-HL-2024",
                  "region": "pan_india", "topic": "loan_details"}
    ),
    Document(
        page_content="Home Loan BFL-HL-2024 eligibility: Age 23–70 years. Salaried: minimum salary ₹40,000/month. "
                     "Self-employed: minimum business vintage 5 years, ITR showing income above ₹6 lakhs per annum. "
                     "CIBIL score 700+ preferred. Co-applicant mandatory for joint property.",
        metadata={"source": "faq_hl_002", "category": "home_loan", "product_code": "BFL-HL-2024",
                  "region": "pan_india", "topic": "eligibility"}
    ),
    Document(
        page_content="Home Loan BFL-HL-2024 documents: Property documents (sale deed, approved building plan, "
                     "NOC from society), KYC documents (Aadhaar, PAN), income proof (salary slips / ITR), "
                     "bank statements last 12 months, property insurance mandatory before disbursement.",
        metadata={"source": "faq_hl_003", "category": "home_loan", "product_code": "BFL-HL-2024",
                  "region": "pan_india", "topic": "documents"}
    ),
    Document(
        page_content="Home Loan BFL-HL-2024 EMI example: ₹50 lakh loan at 8.75% for 240 months. "
                     "EMI = ₹44,295 per month. Total interest = ₹56,30,800. "
                     "Under Section 80C: principal repayment deductible up to ₹1.5 lakh. "
                     "Under Section 24(b): interest deductible up to ₹2 lakh per year.",
        metadata={"source": "faq_hl_004", "category": "home_loan", "product_code": "BFL-HL-2024",
                  "region": "pan_india", "topic": "emi_calculation"}
    ),

    # ── EMI / REPAYMENT ──────────────────────────────────────────
    Document(
        page_content="How to pay EMI on Bajaj Finance loan: Bajaj Finserv App (NACH auto-debit), "
                     "Net banking NEFT/RTGS to Bajaj Finance virtual account, "
                     "UPI using loan account number@bajajfinance, "
                     "Cheque/DD at nearest Bajaj Finance branch. "
                     "EMI due date is same date every month as first EMI. Auto-debit happens at 9 AM.",
        metadata={"source": "faq_emi_001", "category": "emi", "product_code": "ALL",
                  "region": "pan_india", "topic": "payment_methods"}
    ),
    Document(
        page_content="EMI deferment / moratorium policy: Bajaj Finance offers EMI deferment for customers "
                     "facing genuine financial hardship (job loss, medical emergency, natural disaster). "
                     "Maximum deferment period: 3 months. Interest continues to accrue during deferment. "
                     "Deferred EMIs are added to remaining tenure. Apply via helpdesk or app. "
                     "Credit score is NOT impacted during RBI-approved moratorium periods.",
        metadata={"source": "faq_emi_002", "category": "emi", "product_code": "ALL",
                  "region": "pan_india", "topic": "moratorium"}
    ),
    Document(
        page_content="Can I change my EMI due date? Yes, Bajaj Finance allows one EMI due date change "
                     "per loan lifetime. Request must be made at least 15 days before current due date. "
                     "New date must be between 1st and 28th of the month. "
                     "Visit nearest branch or call 8698010101 with loan account number.",
        metadata={"source": "faq_emi_003", "category": "emi", "product_code": "ALL",
                  "region": "pan_india", "topic": "emi_date_change"}
    ),
    Document(
        page_content="EMI bounce handling: If NACH debit fails, Bajaj Finance retries auto-debit twice "
                     "in the same month. Bounce charge ₹700 + GST per bounce. "
                     "You will receive SMS + email notification immediately. "
                     "Pay overdue EMI within 3 days via UPI or net banking to avoid penal interest. "
                     "Consistent bounces may lead to CIBIL downgrade and loan recall notice.",
        metadata={"source": "faq_emi_004", "category": "emi", "product_code": "ALL",
                  "region": "pan_india", "topic": "emi_bounce"}
    ),
    Document(
        page_content="Loan account BFL2024001 details for customer Rahul Sharma: "
                     "Loan type: Personal Loan BFL-PL-2024. Sanctioned amount: ₹5,00,000. "
                     "EMI: ₹16,607. Due date: 5th of every month. "
                     "Outstanding principal: ₹4,23,000. Remaining tenure: 30 months. "
                     "NACH mandate status: Active. Last payment: Successful.",
        metadata={"source": "faq_emi_005", "category": "emi", "product_code": "BFL2024001",
                  "region": "west_india", "topic": "account_details"}
    ),
    Document(
        page_content="EMI double deduction issue: If EMI has been deducted twice from your bank account, "
                     "raise a dispute within 7 days via Bajaj Finserv App > Raise a Request > Duplicate EMI. "
                     "Attach bank statement screenshot showing both debits. "
                     "Refund processed within 5–7 working days to source bank account. "
                     "Contact grievance helpdesk: grievance@bajajfinserv.in or 8698010101.",
        metadata={"source": "faq_emi_006", "category": "emi", "product_code": "ALL",
                  "region": "pan_india", "topic": "double_deduction"}
    ),

    # ── CREDIT CARDS ─────────────────────────────────────────────
    Document(
        page_content="Bajaj Finserv RBL Bank Credit Card BFL-CC-2024: Credit limit ₹50,000 to ₹10,00,000. "
                     "Annual fee: ₹500 (waived on ₹50,000 annual spend). Reward points: 2 points per ₹100. "
                     "Interest rate on outstanding: 3.49% per month (41.88% per annum). "
                     "Minimum payment due: 5% of outstanding or ₹200, whichever is higher.",
        metadata={"source": "faq_cc_001", "category": "credit_card", "product_code": "BFL-CC-2024",
                  "region": "pan_india", "topic": "card_details"}
    ),
    Document(
        page_content="How to close Bajaj Finserv Credit Card BFL-CC-2024: Ensure zero outstanding balance. "
                     "Call 8698010101 or visit branch. Wait 7 working days for closure processing. "
                     "You will receive a No Dues Certificate (NDC) via email within 15 days. "
                     "Cut the card diagonally after receiving NDC. Rewards points expire on closure.",
        metadata={"source": "faq_cc_002", "category": "credit_card", "product_code": "BFL-CC-2024",
                  "region": "pan_india", "topic": "card_closure"}
    ),
    Document(
        page_content="Credit card BFL-CC-2024 reward points redemption: 1 reward point = ₹0.25. "
                     "Minimum redemption: 500 points (= ₹125). Redeem via Bajaj Finserv App > Rewards. "
                     "Points valid for 2 years from date of earning. "
                     "Accelerated points: 5X points on online shopping, 3X on groceries, 2X on fuel (max ₹200/month).",
        metadata={"source": "faq_cc_003", "category": "credit_card", "product_code": "BFL-CC-2024",
                  "region": "pan_india", "topic": "rewards"}
    ),

    # ── FIXED DEPOSITS ───────────────────────────────────────────
    Document(
        page_content="Bajaj Finance Fixed Deposit BFL-FD-2024: Interest rates up to 8.85% per annum. "
                     "Tenure: 12 to 60 months. Minimum deposit: ₹15,000. Maximum: no limit. "
                     "Senior citizens get additional 0.25% interest. "
                     "Interest payout options: Monthly, Quarterly, Half-yearly, Yearly, or Cumulative. "
                     "FAAA/Stable rating by CRISIL. Covered under DICGC up to ₹5 lakhs.",
        metadata={"source": "faq_fd_001", "category": "fixed_deposit", "product_code": "BFL-FD-2024",
                  "region": "pan_india", "topic": "fd_details"}
    ),
    Document(
        page_content="Fixed Deposit BFL-FD-2024 premature withdrawal: Allowed after 3 months (lock-in period). "
                     "Penalty: 1% reduction in applicable interest rate. "
                     "No penalty for premature withdrawal after 18 months for senior citizens. "
                     "TDS deducted at 10% if annual interest exceeds ₹40,000 (₹50,000 for senior citizens). "
                     "Submit Form 15G/15H to avoid TDS if income below taxable limit.",
        metadata={"source": "faq_fd_002", "category": "fixed_deposit", "product_code": "BFL-FD-2024",
                  "region": "pan_india", "topic": "premature_withdrawal"}
    ),

    # ── GOLD LOANS ───────────────────────────────────────────────
    Document(
        page_content="Bajaj Finance Gold Loan BFL-GL-2024: Loan against gold jewellery and ornaments. "
                     "LTV ratio: up to 75% of gold market value (as per RBI guidelines). "
                     "Interest rates: 9.50% to 28% per annum depending on tenure. "
                     "Disbursement in 30 minutes. No credit score check required. "
                     "Gold stored in secured vault, insured. Minimum gold weight: 10 grams (18 karat or above).",
        metadata={"source": "faq_gl_001", "category": "gold_loan", "product_code": "BFL-GL-2024",
                  "region": "pan_india", "topic": "loan_details"}
    ),
    Document(
        page_content="Gold Loan BFL-GL-2024 auction notice: If gold loan EMI is overdue by 90 days, "
                     "Bajaj Finance issues auction notice as per RBI guidelines. "
                     "Customer gets 14 days to clear dues before gold is auctioned. "
                     "Any surplus from auction is returned to customer. "
                     "To prevent auction: immediately contact helpdesk or visit nearest branch.",
        metadata={"source": "faq_gl_002", "category": "gold_loan", "product_code": "BFL-GL-2024",
                  "region": "pan_india", "topic": "auction_notice"}
    ),

    # ── KYC / ACCOUNT ────────────────────────────────────────────
    Document(
        page_content="How to update KYC details in Bajaj Finance: Video KYC available on Bajaj Finserv App "
                     "(complete in 10 minutes). Aadhaar-based eKYC (instant). "
                     "For address change: submit new Aadhaar + utility bill via app or branch. "
                     "PAN update: requires physical visit to branch with original PAN. "
                     "Mobile number update: requires Aadhaar OTP verification at branch.",
        metadata={"source": "faq_kyc_001", "category": "kyc", "product_code": "ALL",
                  "region": "pan_india", "topic": "kyc_update"}
    ),
    Document(
        page_content="Bajaj Finance customer grievance redressal: Level 1: Call 8698010101 (8AM–8PM). "
                     "Level 2: Email grievance@bajajfinserv.in (response within 5 working days). "
                     "Level 3: Nodal Officer at nodalofficer@bajajfinserv.in. "
                     "Level 4: RBI Banking Ombudsman at cms.rbi.org.in. "
                     "Include loan account number in all communications for faster resolution.",
        metadata={"source": "faq_kyc_002", "category": "grievance", "product_code": "ALL",
                  "region": "pan_india", "topic": "grievance_process"}
    ),
    Document(
        page_content="Bajaj Finance account statement download: Bajaj Finserv App > My Loans > Account Statement. "
                     "Select date range (up to 5 years). Format: PDF or Excel. "
                     "Email statement: Login to bajajfinserv.in > Statements > Request. "
                     "Physical statement: ₹50 charge per statement, delivered in 7 working days. "
                     "Amortization schedule available for all active loans.",
        metadata={"source": "faq_kyc_003", "category": "account", "product_code": "ALL",
                  "region": "pan_india", "topic": "statement"}
    ),

    # ── INSURANCE ────────────────────────────────────────────────
    Document(
        page_content="Bajaj Allianz Life Insurance policy BFL-INS-TERM-2024: Term insurance linked to loans. "
                     "Coverage: Outstanding loan amount + 10% buffer. "
                     "Premium: 0.5% to 1.2% of loan amount per annum based on age and health profile. "
                     "Death benefit pays off entire outstanding loan. Family gets no debt burden. "
                     "Critical illness rider available at additional 0.3% premium.",
        metadata={"source": "faq_ins_001", "category": "insurance", "product_code": "BFL-INS-TERM-2024",
                  "region": "pan_india", "topic": "term_insurance"}
    ),

    # ── CIBIL / CREDIT SCORE ─────────────────────────────────────
    Document(
        page_content="How CIBIL score affects Bajaj Finance loan eligibility: "
                     "750–900: Best rates, instant approval. 700–749: Standard rates, quick approval. "
                     "650–699: Higher interest, additional documents may be required. "
                     "Below 650: Application likely rejected. Can apply with co-applicant having 700+ score. "
                     "Check your CIBIL score free once per year at cibil.com.",
        metadata={"source": "faq_cibil_001", "category": "credit_score", "product_code": "ALL",
                  "region": "pan_india", "topic": "cibil_impact"}
    ),
    Document(
        page_content="How to improve CIBIL score for Bajaj Finance loan: Pay all EMIs on time (most important factor, 35% weightage). "
                     "Keep credit utilization below 30% of limit. Do not apply for multiple loans simultaneously. "
                     "Maintain a mix of secured (home/car) and unsecured (personal/credit card) credit. "
                     "CIBIL score improvement typically takes 6–12 months of disciplined repayment.",
        metadata={"source": "faq_cibil_002", "category": "credit_score", "product_code": "ALL",
                  "region": "pan_india", "topic": "cibil_improvement"}
    ),

    # ── DIGITAL / APP ────────────────────────────────────────────
    Document(
        page_content="Bajaj Finserv App features: View all loans and EMI schedule, "
                     "one-click EMI payment via UPI/NetBanking, raise service requests, "
                     "download statements and NOC, check credit score (free monthly), "
                     "apply for new loans and credit cards, invest in FDs, "
                     "manage insurance policies. Available on Android 6.0+ and iOS 13+.",
        metadata={"source": "faq_app_001", "category": "digital", "product_code": "ALL",
                  "region": "pan_india", "topic": "app_features"}
    ),
    Document(
        page_content="Bajaj Finserv App login issues: If OTP not received, check if mobile number is registered. "
                     "OTP expires in 3 minutes — request new OTP. "
                     "If account locked after 5 failed attempts, wait 24 hours or call 8698010101. "
                     "For biometric login issues, disable and re-enable in App > Settings > Security. "
                     "Minimum app version required: v10.5 (update from Play Store / App Store).",
        metadata={"source": "faq_app_002", "category": "digital", "product_code": "ALL",
                  "region": "pan_india", "topic": "app_login"}
    ),

    # ── LOAN AGAINST PROPERTY ────────────────────────────────────
    Document(
        page_content="Bajaj Finance Loan Against Property BFL-LAP-2024: Loan up to ₹5 crores against residential "
                     "or commercial property. LTV: up to 60% of property value. "
                     "Interest rate: 9.75% to 15% per annum. Tenure: 2 to 15 years. "
                     "Both self-occupied and rented properties eligible. "
                     "Property must be free of encumbrances and have clear title.",
        metadata={"source": "faq_lap_001", "category": "lap", "product_code": "BFL-LAP-2024",
                  "region": "pan_india", "topic": "loan_details"}
    ),

    # ── TWO-WHEELER LOANS ────────────────────────────────────────
    Document(
        page_content="Bajaj Finance Two-Wheeler Loan BFL-TWL-2024: Finance up to 100% on-road price. "
                     "Interest rate: 0% to 15% per annum (0% schemes on select models). "
                     "Tenure: 6 to 48 months. Down payment as low as ₹999 on select schemes. "
                     "Approval in 5 minutes. Available at 5000+ dealerships across India. "
                     "Electric two-wheelers eligible with additional green mobility subsidy.",
        metadata={"source": "faq_twl_001", "category": "two_wheeler_loan", "product_code": "BFL-TWL-2024",
                  "region": "pan_india", "topic": "loan_details"}
    ),

    # ── NEAR-DUPLICATES (for reranking demo) ─────────────────────
    Document(
        page_content="Personal loan processing fee at Bajaj Finance: The processing fee for personal loan is "
                     "3.93% of the disbursed loan amount. This fee is deducted upfront from the loan amount "
                     "before credit to your account. GST of 18% applies on the processing fee. "
                     "Example: ₹5 lakh loan → processing fee = ₹19,650 + GST ₹3,537 = ₹23,187 deducted.",
        metadata={"source": "faq_pl_009", "category": "personal_loan", "product_code": "BFL-PL-2024",
                  "region": "pan_india", "topic": "processing_fee"}
    ),
    Document(
        page_content="What is the processing charge on Bajaj Finance personal loan? "
                     "Processing charge is 3.93% of loan amount (inclusive of taxes per latest circular). "
                     "This is a one-time non-refundable fee charged at loan origination. "
                     "Bajaj Finance does not charge any hidden fees beyond the processing charge and GST. "
                     "Zero processing fee offers available during festive seasons — check bajajfinserv.in.",
        metadata={"source": "faq_pl_010", "category": "personal_loan", "product_code": "BFL-PL-2024",
                  "region": "pan_india", "topic": "processing_fee"}
    ),

    # ── REGULATORY ───────────────────────────────────────────────
    Document(
        page_content="RBI Fair Practices Code compliance at Bajaj Finance: Loan sanction letter issued within "
                     "7 days of approval. KFS (Key Fact Statement) provided before loan signing. "
                     "MITC (Most Important Terms and Conditions) document available on website. "
                     "Annual Percentage Rate (APR) clearly disclosed in all loan documents. "
                     "No door-step recovery agents without prior written notice to borrower.",
        metadata={"source": "faq_reg_001", "category": "regulatory", "product_code": "ALL",
                  "region": "pan_india", "topic": "rbi_compliance"}
    ),
    Document(
        page_content="NPA classification at Bajaj Finance (per RBI norms): "
                     "If EMI overdue > 90 days: loan classified as Non-Performing Asset (NPA). "
                     "NPA classification affects CIBIL score severely (can drop 100+ points). "
                     "Recovery process initiates after NPA classification. "
                     "Loan restructuring available before NPA: call helpdesk immediately if struggling to pay.",
        metadata={"source": "faq_reg_002", "category": "regulatory", "product_code": "ALL",
                  "region": "pan_india", "topic": "npa"}
    ),

    # ── BRANCH / SERVICE ─────────────────────────────────────────
    Document(
        page_content="Bajaj Finance branch locator: bajajfinserv.in/branch-locator or App > Find Branch. "
                     "3,000+ branches across 1,600+ cities. "
                     "Branch working hours: Monday–Friday 9:30 AM to 5:30 PM, Saturday 9:30 AM to 1:30 PM. "
                     "Closed on national holidays. "
                     "Customer care: 8698010101 (8 AM – 8 PM, 7 days a week including holidays).",
        metadata={"source": "faq_svc_001", "category": "service", "product_code": "ALL",
                  "region": "pan_india", "topic": "branch_info"}
    ),
    Document(
        page_content="Bajaj Finance NOC (No Objection Certificate): Issued after full loan repayment. "
                     "Auto-generated and emailed within 7 working days of final EMI clearance. "
                     "Download from App > My Loans > Closed Loans > Download NOC. "
                     "For property/vehicle hypothecation removal: NOC + RC form (Form 35) required at RTO. "
                     "Physical NOC (stamped): ₹100 charge, delivered in 15 working days.",
        metadata={"source": "faq_svc_002", "category": "service", "product_code": "ALL",
                  "region": "pan_india", "topic": "noc"}
    ),

    # ── ADDITIONAL CHUNKS to reach 50 ────────────────────────────
    Document(
        page_content="Bajaj Finance Flexi Loan BFL-FLEXI-2024: Withdraw money as needed from approved limit. "
                     "Pay interest only on amount withdrawn, not on entire limit. "
                     "Repay and re-withdraw up to sanctioned limit without re-applying. "
                     "Limit: ₹1 lakh to ₹25 lakhs. Interest rate: 13% to 21% per annum. "
                     "Best for: home renovation, medical emergencies, business working capital.",
        metadata={"source": "faq_flexi_001", "category": "flexi_loan", "product_code": "BFL-FLEXI-2024",
                  "region": "pan_india", "topic": "loan_details"}
    ),
    Document(
        page_content="Bajaj Finance personal loan insurance (BLIP): Bajaj Loan Insurance Plan covers "
                     "outstanding loan amount in case of death or permanent disability. "
                     "Single premium paid upfront added to loan amount. "
                     "Premium: 2.5%–4.5% of loan amount depending on profile. "
                     "Nominee receives full outstanding amount on claim. Highly recommended for sole breadwinners.",
        metadata={"source": "faq_ins_002", "category": "insurance", "product_code": "BFL-BLIP-2024",
                  "region": "pan_india", "topic": "loan_insurance"}
    ),
    Document(
        page_content="Bajaj Finance top-up loan eligibility: Available after 6 months of regular repayment. "
                     "No additional documents required (existing KYC used). "
                     "Top-up amount: up to original loan amount or ₹25 lakhs, whichever is lower. "
                     "Interest rate on top-up: 0.5% higher than existing loan rate. "
                     "Disbursement in 24 hours for eligible customers via pre-approved offer in app.",
        metadata={"source": "faq_pl_011", "category": "personal_loan", "product_code": "BFL-PL-TOPUP",
                  "region": "pan_india", "topic": "top_up"}
    ),
    Document(
        page_content="Bajaj Finance loan restructuring policy: Available for customers with temporary financial difficulty. "
                     "Options: tenure extension (reduces EMI), rate concession (for 3 months), "
                     "EMI holiday (interest accrues, added to principal). "
                     "Apply before NPA classification (before 90 days overdue). "
                     "Submit application to branch with proof of financial hardship (lay-off letter, medical bills).",
        metadata={"source": "faq_reg_003", "category": "regulatory", "product_code": "ALL",
                  "region": "pan_india", "topic": "restructuring"}
    ),
    Document(
        page_content="Bajaj Finance Experia portal bajajfinserv.in features for existing customers: "
                     "View loan account summary, download statements, update KYC, "
                     "submit service requests, track loan application status, "
                     "view pre-approved offers, generate repayment schedule. "
                     "Login: registered mobile OTP or email + password. 24x7 access.",
        metadata={"source": "faq_app_003", "category": "digital", "product_code": "ALL",
                  "region": "pan_india", "topic": "portal_features"}
    ),
    Document(
        page_content="Bajaj Finance e-mandate / NACH registration: Auto-debit mandate registered at loan disbursement. "
                     "Bank account must have sufficient balance on EMI due date. "
                     "To change mandate bank account: visit branch 15 days before next EMI. "
                     "NACH debit happens between 7–9 AM on due date. "
                     "Inward return charges from bank (₹300–700) are passed to customer.",
        metadata={"source": "faq_emi_007", "category": "emi", "product_code": "ALL",
                  "region": "pan_india", "topic": "nach_mandate"}
    ),
    Document(
        page_content="Bajaj Finance loan foreclosure letter request: Get foreclosure letter to know exact amount to close loan. "
                     "Request via App > My Loans > Foreclosure Letter. Valid for 10 days from issue date. "
                     "Pay exact foreclosure amount via NEFT/RTGS to Bajaj Finance virtual account. "
                     "NOC issued within 7 working days of foreclosure payment clearance. "
                     "Foreclosure charge: 4.72% + GST on outstanding principal (waived in some offers).",
        metadata={"source": "faq_pl_012", "category": "personal_loan", "product_code": "BFL-PL-2024",
                  "region": "pan_india", "topic": "foreclosure_process"}
    ),
    Document(
        page_content="Bajaj Finance co-applicant policy for personal loan: Co-applicant can be spouse, parent, or sibling. "
                     "Co-applicant income is clubbed for higher loan eligibility. "
                     "Both applicant and co-applicant equally liable for repayment. "
                     "Loan reflects on CIBIL of both applicant and co-applicant. "
                     "Co-applicant cannot be removed after loan disbursement without court order.",
        metadata={"source": "faq_pl_013", "category": "personal_loan", "product_code": "BFL-PL-2024",
                  "region": "pan_india", "topic": "co_applicant"}
    ),
    Document(
        page_content="Bajaj Finance savings account — not offered directly. Bajaj Finance is an NBFC (Non-Banking Financial Company), "
                     "not a bank. It does not offer savings accounts or current accounts. "
                     "For banking products: use Bajaj Finserv's partner banks (ICICI, Axis, RBL). "
                     "Bajaj Finance offers: loans, fixed deposits, insurance, credit cards (via RBL Bank), "
                     "mutual fund investments via Bajaj Finserv AMC.",
        metadata={"source": "faq_gen_001", "category": "general", "product_code": "ALL",
                  "region": "pan_india", "topic": "company_info"}
    ),
    Document(
        page_content="Bajaj Finance car loan BFL-CAR-2024: New car financing up to 100% on-road. "
                     "Used car financing up to 85% of valuation. "
                     "Interest rate: 7.5% to 14% per annum. Tenure: 12 to 84 months. "
                     "Tie-up with 5,000+ dealerships. Loan against car for existing car owners also available. "
                     "Comprehensive car insurance bundled at preferential rate.",
        metadata={"source": "faq_car_001", "category": "car_loan", "product_code": "BFL-CAR-2024",
                  "region": "pan_india", "topic": "loan_details"}
    ),
]

print(f"✅ Corpus built: {len(bajaj_docs)} documents")
print(f"\nCategories:")
from collections import Counter
cats = Counter(d.metadata['category'] for d in bajaj_docs)
for cat, count in sorted(cats.items()):
    print(f"  {cat}: {count} chunks")
print(f"\nProduct codes: {len(set(d.metadata['product_code'] for d in bajaj_docs))} unique")
print(f"Sample chunk length: {len(bajaj_docs[0].page_content)} chars")

✅ Corpus built: 51 documents

Categories:
  account: 1 chunks
  car_loan: 1 chunks
  credit_card: 3 chunks
  credit_score: 2 chunks
  digital: 3 chunks
  emi: 7 chunks
  fixed_deposit: 2 chunks
  flexi_loan: 1 chunks
  general: 1 chunks
  gold_loan: 2 chunks
  grievance: 1 chunks
  home_loan: 4 chunks
  insurance: 2 chunks
  kyc: 1 chunks
  lap: 1 chunks
  personal_loan: 13 chunks
  regulatory: 3 chunks
  service: 2 chunks
  two_wheeler_loan: 1 chunks

Product codes: 16 unique
Sample chunk length: 257 chars


In [15]:
len(bajaj_docs[0].page_content)

257

## 🎯 Step 2 — Ground Truth for Evaluation

To measure Precision@K, Recall@K, MRR — we need to know which chunks are **truly relevant** for each test query.  
In production you'd collect this from human annotators. For demo, we define it manually.


In [16]:
# Ground truth: {query: set of relevant source IDs}
# These are the source IDs from metadata above

GROUND_TRUTH = {
    "What is the EMI for personal loan BFL-PL-2024?": {
        "faq_pl_004",   # EMI calculation example
        "faq_emi_005",  # BFL2024001 account with ₹16,607 EMI
        "faq_pl_001",   # loan details with tenure/amount
    },
    "What documents are required for personal loan?": {
        "faq_pl_003",   # documents for BFL-PL-2024
        "faq_kyc_001",  # KYC documents
    },
    "How to foreclose my Bajaj Finance loan?": {
        "faq_pl_007",   # foreclosure charges and rules
        "faq_pl_012",   # foreclosure letter process
    },
    "What is the processing fee for personal loan BFL-PL-2024?": {
        "faq_pl_009",   # processing fee detailed
        "faq_pl_010",   # processing charge alternative wording
        "faq_pl_001",   # mentions 3.93% processing fee
    },
    "Can I defer my EMI payment?": {
        "faq_emi_002",  # moratorium/deferment policy
        "faq_reg_003",  # restructuring options
    },
    "EMI double deduction refund process": {
        "faq_emi_006",  # duplicate EMI dispute
        "faq_kyc_002",  # grievance process
    },
    "BFL-PL-2024-MUMBAI interest rate": {
        "faq_pl_005",   # Mumbai regional offer
    },
    "Fixed deposit interest rate senior citizen": {
        "faq_fd_001",   # FD details with senior citizen rate
        "faq_fd_002",   # premature withdrawal + senior citizen TDS
    },
    "How to improve CIBIL score?": {
        "faq_cibil_002",  # CIBIL improvement tips
        "faq_cibil_001",  # CIBIL impact on eligibility
    },
    "What happens if EMI bounces?": {
        "faq_emi_004",   # bounce charges and retry
        "faq_pl_008",    # late payment charges
        "faq_emi_007",   # NACH mandate details
    },
}

print(f"✅ Ground truth defined for {len(GROUND_TRUTH)} test queries")
for q, rel in GROUND_TRUTH.items():
    print(f"  Q: '{q[:55]}...' → {len(rel)} relevant docs: {rel}")

✅ Ground truth defined for 10 test queries
  Q: 'What is the EMI for personal loan BFL-PL-2024?...' → 3 relevant docs: {'faq_pl_004', 'faq_pl_001', 'faq_emi_005'}
  Q: 'What documents are required for personal loan?...' → 2 relevant docs: {'faq_kyc_001', 'faq_pl_003'}
  Q: 'How to foreclose my Bajaj Finance loan?...' → 2 relevant docs: {'faq_pl_007', 'faq_pl_012'}
  Q: 'What is the processing fee for personal loan BFL-PL-202...' → 3 relevant docs: {'faq_pl_009', 'faq_pl_010', 'faq_pl_001'}
  Q: 'Can I defer my EMI payment?...' → 2 relevant docs: {'faq_reg_003', 'faq_emi_002'}
  Q: 'EMI double deduction refund process...' → 2 relevant docs: {'faq_kyc_002', 'faq_emi_006'}
  Q: 'BFL-PL-2024-MUMBAI interest rate...' → 1 relevant docs: {'faq_pl_005'}
  Q: 'Fixed deposit interest rate senior citizen...' → 2 relevant docs: {'faq_fd_001', 'faq_fd_002'}
  Q: 'How to improve CIBIL score?...' → 2 relevant docs: {'faq_cibil_001', 'faq_cibil_002'}
  Q: 'What happens if EMI bounces?...' → 3 relevant

## 🔵 Step 3 — Build ChromaDB Dense Index

**What this is:** Encodes all 50 chunks using `all-MiniLM-L6-v2` and stores them in ChromaDB (local, persisted to disk).  
**Why ChromaDB:** It's LangChain-native, runs fully locally, no API key.  
**This replaces Session 3's FAISS build** — same concept, managed vectorstore.


In [ ]:
import os
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Use HuggingFace embeddings — same all-MiniLM-L6-v2 from Sessions 2 & 3
# FREE — no OpenAI API key needed
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}  # L2-normalize for cosine via dot product
)

CHROMA_DIR = "./bajajbot_chroma_s4"

# Build ChromaDB — encodes all 50 chunks and persists to disk
print("Building ChromaDB index... (encodes 50 chunks)")
chroma_db = Chroma.from_documents(
    documents=bajaj_docs,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
    collection_name="bajajbot_faq"
)
print(f"✅ ChromaDB built! {chroma_db._collection.count()} vectors stored")
print(f"   Persisted at: {CHROMA_DIR}/")



/var/folders/fq/kr6gv5l17pd4j572n0_n3f2c0000gn/T/ipykernel_58160/2336415698.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Building ChromaDB index... (encodes 50 chunks)
✅ ChromaDB built! 51 vectors stored
   Persisted at: ./bajajbot_chroma_s4/

--- Dense retrieval test ---
Query: 'Can I defer my EMI payment?'
  1. [faq_emi_002] EMI deferment / moratorium policy: Bajaj Finance offers EMI deferment for custom...
  2. [faq_emi_006] EMI double deduction issue: If EMI has been deducted twice from your bank accoun...
  3. [faq_emi_003] Can I change my EMI due date? Yes, Bajaj Finance allows one EMI due date change ...
  4. [faq_emi_004] EMI bounce handling: If NACH debit fails, Bajaj Finance retries auto-debit twice...
  5. [faq_pl_008] Personal Loan BFL-PL-2024 late payment charges: If EMI is not paid by due date, ...


In [20]:
# Create dense retriever
dense_retriever = chroma_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
    score_threshold=0.75  # Only return results with cosine similarity >= 0.75 (after normalization)
)

# Quick test
print("\n--- Dense retrieval test ---")
test_query = "Can I defer my EMI payment?"
results = dense_retriever.invoke(test_query)
print(f"Query: '{test_query}'")
for i, doc in enumerate(results, 1):
    print(f"  {i}. [{doc.metadata['source']}] {doc.page_content[:80]}...")


--- Dense retrieval test ---
Query: 'Can I defer my EMI payment?'
  1. [faq_emi_002] EMI deferment / moratorium policy: Bajaj Finance offers EMI deferment for custom...
  2. [faq_emi_006] EMI double deduction issue: If EMI has been deducted twice from your bank accoun...
  3. [faq_emi_003] Can I change my EMI due date? Yes, Bajaj Finance allows one EMI due date change ...


In [22]:
# results[]

## 🟠 Step 4 — Build BM25 Sparse Retriever

**Why sparse matters:** Dense search will FAIL on exact product codes like `BFL-PL-2024-MUMBAI`.  
The embedding model never saw this string in training — it has no meaningful vector.  
BM25 finds it instantly via exact token matching.


In [ ]:
from langchain_community.retrievers import BM25Retriever

sparse_retriever = BM25Retriever.from_documents(bajaj_docs)
sparse_retriever.k = 5



print("✅ BM25 sparse retriever built")

# ── TEST 1: Exact product code — DENSE FAILS, SPARSE WINS ──────
print("\n=== TEST: Product Code ===")
query_code = "BFL-PL-2024 interest rate"

dense_results  = dense_retriever.invoke(query_code)
sparse_results = sparse_retriever.invoke(query_code)

for i, doc in enumerate(sparse_results, 1):
    match = "✅" if doc.metadata['source'] == 'faq_pl_005' else "  "
    print(f"  {i}. {match} [{doc.metadata['source']}] {doc.page_content[:75]}...")



for i, doc in enumerate(dense_results, 1):
    match = "✅" if doc.metadata['source'] == 'faq_pl_005' else "  "
    print(f"  {i}. {match} [{doc.metadata['source']}] {doc.page_content[:75]}...")


✅ BM25 sparse retriever built

=== TEST: Product Code ===
  1.    [faq_pl_004] Personal Loan BFL-PL-2024 EMI calculation example: For ₹5,00,000 loan at 12...
  2.    [faq_pl_008] Personal Loan BFL-PL-2024 late payment charges: If EMI is not paid by due d...
  3.    [faq_hl_004] Home Loan BFL-HL-2024 EMI example: ₹50 lakh loan at 8.75% for 240 months. E...
  4.    [faq_fd_002] Fixed Deposit BFL-FD-2024 premature withdrawal: Allowed after 3 months (loc...
  5.    [faq_pl_002] Bajaj Finance Personal Loan BFL-PL-2024 eligibility criteria: Minimum age 2...
  1. ✅ [faq_pl_005] Bajaj Finance Personal Loan BFL-PL-2024-MUMBAI: Special Mumbai zone offer. ...
  2.    [faq_pl_006] Personal Loan BFL-PL-2024-DELHI: Delhi NCR special offer at 10.75% per annu...
  3.    [faq_pl_004] Personal Loan BFL-PL-2024 EMI calculation example: For ₹5,00,000 loan at 12...


In [33]:

# ── TEST 2: Semantic query — DENSE WINS ────────────────────────
print("\n=== TEST: Semantic query (dense wins) ===")
query_semantic = "What happens if I can't pay my loan installment this month?"

dense_results2  = dense_retriever.invoke(query_semantic)
sparse_results2 = sparse_retriever.invoke(query_semantic)

print(f"\nQuery: '{query_semantic}'")
print("\nDense results (finds 'moratorium', 'deferment', 'EMI holiday'):")
for i, doc in enumerate(dense_results2, 1):
    print(f"  {i}. [{doc.metadata['source']}] [{doc.metadata['topic']}] {doc.page_content[:70]}...")

print("\nSparse results (only finds exact words 'pay', 'loan'):")
for i, doc in enumerate(sparse_results2, 1):
    print(f"  {i}. [{doc.metadata['source']}] [{doc.metadata['topic']}] {doc.page_content[:70]}...")


=== TEST: Semantic query (dense wins) ===

Query: 'What happens if I can't pay my loan installment this month?'

Dense results (finds 'moratorium', 'deferment', 'EMI holiday'):
  1. [faq_pl_008] [late_payment] Personal Loan BFL-PL-2024 late payment charges: If EMI is not paid by ...
  2. [faq_reg_003] [restructuring] Bajaj Finance loan restructuring policy: Available for customers with ...
  3. [faq_reg_002] [npa] NPA classification at Bajaj Finance (per RBI norms): If EMI overdue > ...

Sparse results (only finds exact words 'pay', 'loan'):
  1. [faq_emi_003] [emi_date_change] Can I change my EMI due date? Yes, Bajaj Finance allows one EMI due da...
  2. [faq_emi_001] [payment_methods] How to pay EMI on Bajaj Finance loan: Bajaj Finserv App (NACH auto-deb...
  3. [faq_pl_006] [regional_offer] Personal Loan BFL-PL-2024-DELHI: Delhi NCR special offer at 10.75% per...
  4. [faq_pl_010] [processing_fee] What is the processing charge on Bajaj Finance personal loan? Processi...
  5. [faq_e

## 🟢 Step 5 — Hybrid Search (EnsembleRetriever)

**LangChain EnsembleRetriever** = BM25 (sparse) + ChromaDB (dense) with weighted combination.  
`weights=[0.3, 0.7]` → 30% keyword matching + 70% semantic meaning.  
**This is your alpha parameter from the theory session.**


In [34]:
from langchain_classic.retrievers import EnsembleRetriever

# ── BUILD HYBRID RETRIEVER ──────────────────────────────────────
hybrid_retriever = EnsembleRetriever(
    retrievers=[sparse_retriever, dense_retriever],
    weights=[0.3, 0.7]   # 30% sparse (BM25) + 70% dense (ChromaDB)
)
print("✅ Hybrid retriever built (BM25 30% + ChromaDB 70%)")

# ── TEST: Query that needs BOTH ─────────────────────────────────
# 'interest rate' = semantic concept (dense handles)
# 'BFL-PL-2024-MUMBAI' = exact product code (sparse handles)

query_hybrid = "What is the interest rate for personal loan BFL-PL-2024-MUMBAI?"
print(f"\nQuery: '{query_hybrid}'")

print("\n--- Dense only (misses the Mumbai product code) ---")
for i, doc in enumerate(dense_retriever.invoke(query_hybrid), 1):
    match = "✅" if 'MUMBAI' in doc.page_content or doc.metadata['source'] == 'faq_pl_005' else "  "
    print(f"  {i}. {match} [{doc.metadata['source']}] {doc.page_content[:80]}...")

print("\n--- Sparse only (misses semantic context) ---")
for i, doc in enumerate(sparse_retriever.invoke(query_hybrid), 1):
    match = "✅" if 'MUMBAI' in doc.page_content or doc.metadata['source'] == 'faq_pl_005' else "  "
    print(f"  {i}. {match} [{doc.metadata['source']}] {doc.page_content[:80]}...")

print("\n--- HYBRID (best of both) ---")
for i, doc in enumerate(hybrid_retriever.invoke(query_hybrid), 1):
    match = "✅" if 'MUMBAI' in doc.page_content or doc.metadata['source'] == 'faq_pl_005' else "  "
    print(f"  {i}. {match} [{doc.metadata['source']}] {doc.page_content[:80]}...")

# ── ALPHA TUNING DEMO ───────────────────────────────────────────
print("\n=== Alpha tuning: how weights change results ===")
for sparse_w, dense_w in [(0.1, 0.9), (0.5, 0.5), (0.9, 0.1)]:
    ret = EnsembleRetriever(
        retrievers=[sparse_retriever, dense_retriever],
        weights=[sparse_w, dense_w]
    )
    results = ret.invoke(query_hybrid)
    sources = [d.metadata['source'] for d in results]
    mumbai_rank = next((i+1 for i, d in enumerate(results) if d.metadata['source'] == 'faq_pl_005'), 'not found')
    print(f"  sparse={sparse_w}, dense={dense_w}: Mumbai doc at rank {mumbai_rank} | Top sources: {sources[:3]}")

✅ Hybrid retriever built (BM25 30% + ChromaDB 70%)

Query: 'What is the interest rate for personal loan BFL-PL-2024-MUMBAI?'

--- Dense only (misses the Mumbai product code) ---
  1. ✅ [faq_pl_005] Bajaj Finance Personal Loan BFL-PL-2024-MUMBAI: Special Mumbai zone offer. Inter...
  2.    [faq_pl_001] Bajaj Finance Personal Loan BFL-PL-2024 offers loan amounts from ₹1 lakh to ₹40 ...
  3.    [faq_pl_006] Personal Loan BFL-PL-2024-DELHI: Delhi NCR special offer at 10.75% per annum. Co...

--- Sparse only (misses semantic context) ---
  1.    [faq_pl_010] What is the processing charge on Bajaj Finance personal loan? Processing charge ...
  2.    [faq_pl_009] Personal loan processing fee at Bajaj Finance: The processing fee for personal l...
  3.    [faq_pl_004] Personal Loan BFL-PL-2024 EMI calculation example: For ₹5,00,000 loan at 12% per...
  4.    [faq_pl_013] Bajaj Finance co-applicant policy for personal loan: Co-applicant can be spouse,...
  5.    [faq_pl_011] Bajaj Finance top-up

## 📊 Step 6 — Retrieval Evaluation Metrics

Now we **measure** whether our retrievers are actually working.  
Using our ground truth from Step 2 — we run each query and compute:
- **Precision@K** — of top-K results, what fraction are relevant?
- **Recall@K** — of all relevant docs, how many did we find?
- **MRR@K** — where does the first relevant doc appear?
- **NDCG@K** — are relevant docs ranked at the top?


In [12]:
import math
import numpy as np

def precision_at_k(ranked_sources, relevant_sources, k):
    topk = ranked_sources[:k]
    hits = sum(1 for s in topk if s in relevant_sources)
    return hits / max(1, k)

def recall_at_k(ranked_sources, relevant_sources, k):
    if not relevant_sources: return 0.0
    topk = ranked_sources[:k]
    hits = sum(1 for s in topk if s in relevant_sources)
    return hits / len(relevant_sources)

def mrr_at_k(ranked_sources, relevant_sources, k):
    for i, s in enumerate(ranked_sources[:k], start=1):
        if s in relevant_sources:
            return 1.0 / i
    return 0.0

def ndcg_at_k(ranked_sources, relevant_sources, k):
    gains  = {s: 1.0 for s in relevant_sources}
    dcg    = sum(gains.get(ranked_sources[i], 0) / math.log2(i + 2)
                 for i in range(min(k, len(ranked_sources))))
    ideal  = sorted(gains.values(), reverse=True)[:k]
    idcg   = sum(g / math.log2(i + 2) for i, g in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_retriever(retriever, ground_truth, k=5, name="Retriever"):
    """Run all queries, compute average metrics across all queries."""
    p_scores, r_scores, mrr_scores, ndcg_scores = [], [], [], []

    print(f"\n{'='*60}")
    print(f"Evaluating: {name} (K={k})")
    print(f"{'='*60}")

    for query, relevant in ground_truth.items():
        results = retriever.invoke(query)
        ranked_sources = [doc.metadata['source'] for doc in results]

        p   = precision_at_k(ranked_sources, relevant, k)
        r   = recall_at_k(ranked_sources, relevant, k)
        mrr = mrr_at_k(ranked_sources, relevant, k)
        ndcg= ndcg_at_k(ranked_sources, relevant, k)

        p_scores.append(p); r_scores.append(r)
        mrr_scores.append(mrr); ndcg_scores.append(ndcg)

        print(f"\nQ: '{query[:55]}...'")
        print(f"   Retrieved: {ranked_sources}")
        print(f"   Relevant:  {relevant}")
        print(f"   P@{k}={p:.2f}  R@{k}={r:.2f}  MRR={mrr:.2f}  NDCG={ndcg:.2f}")

    print(f"\n{'─'*40}")
    print(f"AVERAGE METRICS for {name}:")
    print(f"  Precision@{k}: {np.mean(p_scores):.3f}")
    print(f"  Recall@{k}:    {np.mean(r_scores):.3f}")
    print(f"  MRR@{k}:       {np.mean(mrr_scores):.3f}")
    print(f"  NDCG@{k}:      {np.mean(ndcg_scores):.3f}")

    return {
        "precision": np.mean(p_scores), "recall": np.mean(r_scores),
        "mrr": np.mean(mrr_scores), "ndcg": np.mean(ndcg_scores)
    }

print("✅ Metric functions ready")

✅ Metric functions ready


In [13]:
# Run evaluation on all 3 retrievers
K = 5

dense_metrics  = evaluate_retriever(dense_retriever,  GROUND_TRUTH, k=K, name="Dense (ChromaDB)")
sparse_metrics = evaluate_retriever(sparse_retriever, GROUND_TRUTH, k=K, name="Sparse (BM25)")
hybrid_metrics = evaluate_retriever(hybrid_retriever, GROUND_TRUTH, k=K, name="Hybrid (70% Dense + 30% BM25)")

# ── COMPARISON TABLE ────────────────────────────────────────────
print("\n" + "="*55)
print("RETRIEVER COMPARISON TABLE")
print("="*55)
print(f"{'Metric':<15} {'Dense':>12} {'Sparse':>12} {'Hybrid':>12}")
print("-"*55)
for metric in ["precision", "recall", "mrr", "ndcg"]:
    print(f"{metric.capitalize()+'@'+str(K):<15} "
          f"{dense_metrics[metric]:>12.3f} "
          f"{sparse_metrics[metric]:>12.3f} "
          f"{hybrid_metrics[metric]:>12.3f}")
print("="*55)
print("→ Hybrid should win on most metrics")
print("→ Sparse wins on product-code queries, Dense on semantic")


Evaluating: Dense (ChromaDB) (K=5)

Q: 'What is the EMI for personal loan BFL-PL-2024?...'
   Retrieved: ['faq_pl_004', 'faq_emi_005', 'faq_pl_008', 'faq_hl_004', 'faq_pl_006']
   Relevant:  {'faq_pl_004', 'faq_pl_001', 'faq_emi_005'}
   P@5=0.40  R@5=0.67  MRR=1.00  NDCG=0.77

Q: 'What documents are required for personal loan?...'
   Retrieved: ['faq_pl_003', 'faq_hl_003', 'faq_pl_013', 'faq_pl_002', 'faq_hl_002']
   Relevant:  {'faq_kyc_001', 'faq_pl_003'}
   P@5=0.20  R@5=0.50  MRR=1.00  NDCG=0.61

Q: 'How to foreclose my Bajaj Finance loan?...'
   Retrieved: ['faq_pl_007', 'faq_pl_012', 'faq_reg_003', 'faq_lap_001', 'faq_svc_002']
   Relevant:  {'faq_pl_007', 'faq_pl_012'}
   P@5=0.40  R@5=1.00  MRR=1.00  NDCG=1.00

Q: 'What is the processing fee for personal loan BFL-PL-202...'
   Retrieved: ['faq_pl_001', 'faq_pl_006', 'faq_pl_009', 'faq_pl_008', 'faq_pl_005']
   Relevant:  {'faq_pl_009', 'faq_pl_010', 'faq_pl_001'}
   P@5=0.40  R@5=0.67  MRR=1.00  NDCG=0.70

Q: 'Can I defer my 

## 🎯 Step 7 — Reranking

**Why reranking?** Even the best retriever sometimes puts the truly best chunk at rank 3 or 4.  
Reranking runs a **second, smarter pass** over the top candidates to fix the ordering.

### 7A — BM25 Reranking over ChromaDB candidates


In [ ]:
import re

# ── BM25 SCORER ─────────────────────────────────────────────────
def tokenize(s):
    return re.findall(r'[a-z0-9]+', s.lower())

# Build BM25 structures from corpus
corpus_tokens = [tokenize(doc.page_content) for doc in bajaj_docs]
doc_sources   = [doc.metadata['source'] for doc in bajaj_docs]
doc_lengths   = [len(t) for t in corpus_tokens]
avgdl         = sum(doc_lengths) / len(doc_lengths)

df = {}
for tokens in corpus_tokens:
    for w in set(tokens):
        df[w] = df.get(w, 0) + 1

N, k1, b = len(corpus_tokens), 1.2, 0.75

def idf(w):
    n = df.get(w, 0)
    return math.log((N - n + 0.5) / (n + 0.5) + 1.0)

def bm25_score(query_tokens, doc_idx):
    tokens = corpus_tokens[doc_idx]
    if not tokens: return 0.0
    tf   = {}
    for w in tokens: tf[w] = tf.get(w, 0) + 1
    dl   = len(tokens)
    norm = k1 * (1 - b + b * dl / avgdl)
    return sum(
        idf(w) * (tf.get(w, 0) * (k1 + 1)) / (tf.get(w, 0) + norm)
        for w in query_tokens if tf.get(w)
    )

# ── RERANKING PIPELINE ──────────────────────────────────────────
def rerank_bm25(query, stage1_docs, top_n=5):
    """Stage 1 docs from ChromaDB, reranked by BM25 score."""
    qtoks = tokenize(query)
    scored = []
    for doc in stage1_docs:
        src = doc.metadata['source']
        idx = doc_sources.index(src) if src in doc_sources else -1
        score = bm25_score(qtoks, idx) if idx >= 0 else 0.0
        scored.append((score, doc))
    scored.sort(reverse=True)
    return [doc for _, doc in scored[:top_n]]

# ── DEMO: BEFORE vs AFTER RERANKING ────────────────────────────
query = "What is the processing fee for personal loan BFL-PL-2024?"
relevant = GROUND_TRUTH[query]

# Stage 1: Get top 10 from ChromaDB (wider net)
dense_top10 = chroma_db.as_retriever(search_kwargs={"k": 10}).invoke(query)

# Stage 2: BM25 rerank
reranked = rerank_bm25(query, dense_top10, top_n=5)

print(f"Query: '{query}'")
print(f"Ground truth: {relevant}")

before_sources = [d.metadata['source'] for d in dense_top10[:5]]
after_sources  = [d.metadata['source'] for d in reranked]

print("\n=== BEFORE Reranking (Dense top-5) ===")
for i, doc in enumerate(dense_top10[:5], 1):
    match = "✅" if doc.metadata['source'] in relevant else "  "
    print(f"  {i}. {match} [{doc.metadata['source']}] {doc.page_content[:75]}...")

print("\n=== AFTER BM25 Reranking (top-5) ===")
for i, doc in enumerate(reranked, 1):
    match = "✅" if doc.metadata['source'] in relevant else "  "
    print(f"  {i}. {match} [{doc.metadata['source']}] {doc.page_content[:75]}...")

# Compare MRR
mrr_before = mrr_at_k(before_sources, relevant, 5)
mrr_after  = mrr_at_k(after_sources, relevant, 5)
print(f"\nMRR before reranking: {mrr_before:.3f}")
print(f"MRR after reranking:  {mrr_after:.3f}  {'✅ improved!' if mrr_after > mrr_before else ''}")

### 7B — Bi-Encoder vs Cross-Encoder (Live Comparison)

**This demonstrates exactly why they're different:**
- Bi-encoder: query and doc encoded **separately** → cosine similarity
- Cross-encoder: reads query + doc **together** → direct relevance score


In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np

# ── BI-ENCODER (what you've been using all along) ───────────────
bi_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

query = "What is my EMI for loan BFL2024001?"

# Pick 5 candidate chunks to compare
candidates = [
    bajaj_docs[3],   # faq_pl_004 — EMI calculation example
    bajaj_docs[17],  # faq_emi_005 — BFL2024001 specific account, ₹16,607 EMI
    bajaj_docs[13],  # faq_emi_001 — how to pay EMI
    bajaj_docs[20],  # faq_cc_001 — credit card (irrelevant)
    bajaj_docs[25],  # faq_fd_001 — fixed deposit (irrelevant)
]

candidate_texts = [doc.page_content for doc in candidates]
candidate_sources = [doc.metadata['source'] for doc in candidates]

# Bi-encoder: encode separately, score with cosine
q_emb   = bi_model.encode([query], normalize_embeddings=True)
d_embs  = bi_model.encode(candidate_texts, normalize_embeddings=True)
bi_scores = (d_embs @ q_emb.T).ravel()  # cosine similarity

print("=== BI-ENCODER SCORES (query and doc encoded separately) ===")
print(f"Query: '{query}'\n")
for src, score, text in sorted(zip(candidate_sources, bi_scores, candidate_texts),
                                key=lambda x: x[1], reverse=True):
    print(f"  {score:.4f} | [{src}] {text[:80]}...")

# Cross-encoder: reads query + doc TOGETHER
print("\n=== CROSS-ENCODER SCORES (query + doc read together) ===")
cross_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
pairs = [(query, text) for text in candidate_texts]
cross_scores = cross_model.predict(pairs)

for src, score, text in sorted(zip(candidate_sources, cross_scores, candidate_texts),
                                key=lambda x: x[1], reverse=True):
    print(f"  {score:+.4f} | [{src}] {text[:80]}...")

print("\n=== COMPARISON ===")
print("Notice: Cross-encoder gives a SHARPER gap between relevant and irrelevant.")
print("faq_emi_005 (BFL2024001 specific) should rank #1 in cross-encoder.")
print("faq_fd_001 (fixed deposit, irrelevant) should have very low cross-encoder score.")

## ⚡ Step 8 — QPS and Latency Benchmarking

**Build a FAISS HNSW index** (same as Session 3) and measure:  
- Single-thread QPS and latency  
- p50 / p95 / p99 latency percentiles  
- What happens when you increase nprobe / efSearch


In [ ]:
import faiss
import numpy as np
import time

# ── BUILD EMBEDDINGS FOR ALL DOCS ──────────────────────────────
print("Encoding all 50 chunks for FAISS...")
texts_list = [doc.page_content for doc in bajaj_docs]
emb = bi_model.encode(texts_list, convert_to_numpy=True,
                      normalize_embeddings=True).astype("float32")
N, d = emb.shape
print(f"✅ Embeddings: {N} vectors × {d} dimensions")

# ── BUILD FAISS HNSW ────────────────────────────────────────────
base_index = faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT)
base_index.hnsw.efConstruction = 200
hnsw_index = faiss.IndexIDMap2(base_index)

ids = np.arange(N, dtype=np.int64)
t0 = time.perf_counter()
hnsw_index.add_with_ids(emb, ids)
build_time = time.perf_counter() - t0

print(f"✅ FAISS HNSW built: {hnsw_index.ntotal} vectors in {build_time:.3f}s")
print(f"   Rate: {N/build_time:.0f} vectors/second")
faiss.write_index(hnsw_index, "bajajbot_hnsw.index")
print(f"   Saved to bajajbot_hnsw.index")

In [ ]:
# ── QPS + LATENCY BENCHMARK ─────────────────────────────────────
import numpy as np

# Build realistic query set (same queries repeated = mimics production load)
query_texts_bench = list(GROUND_TRUTH.keys()) * 100   # 1000 queries

# Encode ALL queries once — measure search latency only
print("Encoding query set...")
q_mat = bi_model.encode(query_texts_bench, convert_to_numpy=True,
                         normalize_embeddings=True).astype("float32")
print(f"Query matrix: {q_mat.shape}")

# Warm-up
for i in range(20):
    hnsw_index.search(q_mat[i:i+1], 5)

def benchmark(index, queries, top_k=5, label=""):
    lat_ms = []
    t_start = time.perf_counter()
    for qv in queries:
        t0 = time.perf_counter()
        index.search(qv.reshape(1, -1), top_k)
        lat_ms.append((time.perf_counter() - t0) * 1000)
    total = time.perf_counter() - t_start
    lat_ms = np.array(lat_ms)
    print(f"\n[{label}]")
    print(f"  QPS:       {len(queries)/total:,.0f}")
    print(f"  p50 (ms):  {np.percentile(lat_ms, 50):.3f}")
    print(f"  p95 (ms):  {np.percentile(lat_ms, 95):.3f}")
    print(f"  p99 (ms):  {np.percentile(lat_ms, 99):.3f}")
    return {"qps": len(queries)/total, "p50": np.percentile(lat_ms, 50),
            "p95": np.percentile(lat_ms, 95), "p99": np.percentile(lat_ms, 99)}

print("\n=== QPS + LATENCY BENCHMARK ===")
r1 = benchmark(hnsw_index, q_mat, top_k=5,  label="HNSW top-k=5")
r2 = benchmark(hnsw_index, q_mat, top_k=10, label="HNSW top-k=10")

# Flat exact index comparison
flat_index = faiss.IndexIDMap2(faiss.IndexFlatIP(d))
flat_index.add_with_ids(emb, ids)
r3 = benchmark(flat_index, q_mat, top_k=5, label="Flat (exact) top-k=5")

print("\n=== SUMMARY: HNSW vs Flat ===")
print(f"{'Metric':<12} {'HNSW k=5':>12} {'HNSW k=10':>12} {'Flat k=5':>12}")
print("-"*50)
for metric, v1, v2, v3 in [("QPS", r1['qps'], r2['qps'], r3['qps']),
                             ("p50 (ms)", r1['p50'], r2['p50'], r3['p50']),
                             ("p95 (ms)", r1['p95'], r2['p95'], r3['p95']),
                             ("p99 (ms)", r1['p99'], r2['p99'], r3['p99'])]:
    if metric == "QPS":
        print(f"{metric:<12} {v1:>12,.0f} {v2:>12,.0f} {v3:>12,.0f}")
    else:
        print(f"{metric:<12} {v1:>12.3f} {v2:>12.3f} {v3:>12.3f}")
print("→ HNSW is faster; Flat is exact but slower at scale")

## 🏁 Step 9 — Full BajajBot v0.4 Pipeline

Putting it all together: **Hybrid retrieval → BM25 reranking → LLM answer**


In [ ]:
# NOTE: Set your OpenAI key to run the LLM step
# import os; os.environ["OPENAI_API_KEY"] = "sk-..."
# If no key available — the retrieval + reranking steps still demonstrate the pipeline

from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

def bajajbot_v04(user_query: str, verbose: bool = True):
    """Full BajajBot v0.4 pipeline: Hybrid → Rerank → LLM"""

    if verbose: print(f"\n{'='*60}\n🤖 BajajBot v0.4\nQuery: {user_query}\n{'='*60}")

    # ── STAGE 1: HYBRID RETRIEVAL (top-10 candidates) ──────────
    wide_retriever = EnsembleRetriever(
        retrievers=[
            BM25Retriever.from_documents(bajaj_docs, k=10),
            chroma_db.as_retriever(search_kwargs={"k": 10})
        ],
        weights=[0.3, 0.7]
    )
    candidates = wide_retriever.invoke(user_query)
    if verbose:
        print(f"Stage 1 — Hybrid retrieved {len(candidates)} candidates")
        for i, doc in enumerate(candidates[:3], 1):
            print(f"  {i}. [{doc.metadata['source']}] {doc.page_content[:60]}...")

    # ── STAGE 2: BM25 RERANKING (top-5 final) ──────────────────
    final_docs = rerank_bm25(user_query, candidates, top_n=5)
    if verbose:
        print(f"\nStage 2 — BM25 reranked, top 5:")
        for i, doc in enumerate(final_docs, 1):
            print(f"  {i}. [{doc.metadata['source']}] {doc.page_content[:60]}...")

    # ── STAGE 3: LLM GENERATION ────────────────────────────────
    context = "\n\n".join(
        f"[{doc.metadata['source']}] {doc.page_content}"
        for doc in final_docs
    )

    prompt_template = PromptTemplate(
        input_variables=["context", "question"],
        template="""You are BajajBot, Bajaj Finance's AI helpdesk assistant.
Answer ONLY using the CONTEXT provided below. Be concise and specific.
If the answer is not in CONTEXT, say: "I don't have this information. Please call 8698010101."

CONTEXT:
{context}

CUSTOMER QUESTION: {question}

BAJAJBOT ANSWER:"""
    )

    try:
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        chain = LLMChain(llm=llm, prompt=prompt_template)
        answer = chain.run(context=context, question=user_query)
        if verbose: print(f"\n💬 BajajBot Answer:\n{answer}")
        return answer
    except Exception as e:
        if verbose:
            print(f"\n(No OpenAI key — showing retrieved context only)")
            print(f"Top chunk: {final_docs[0].page_content}")
        return final_docs[0].page_content

# Test it!
bajajbot_v04("What is the processing fee for personal loan BFL-PL-2024?")
bajajbot_v04("BFL-PL-2024-MUMBAI interest rate")
bajajbot_v04("Can I defer my EMI this month due to job loss?")

## ✅ Session 4 Summary — What You Built

| Component | Code | What it does |
|---|---|---|
| **Corpus** | Step 1 | 50 realistic BajajBot FAQ chunks with metadata |
| **Ground Truth** | Step 2 | 10 queries with known relevant docs (for eval) |
| **Dense Index** | Step 3 | ChromaDB + all-MiniLM-L6-v2 (semantic search) |
| **Sparse Index** | Step 4 | BM25Retriever (exact keyword / product code search) |
| **Hybrid** | Step 5 | EnsembleRetriever (70% dense + 30% sparse) |
| **Eval Metrics** | Step 6 | P@K, R@K, MRR, NDCG across all 3 retrievers |
| **Reranking** | Step 7 | BM25 rerank + CrossEncoder live comparison |
| **QPS/Latency** | Step 8 | FAISS HNSW benchmark, p50/p95/p99 |
| **Full Pipeline** | Step 9 | Hybrid → Rerank → LLM = BajajBot v0.4 |

**Key insight from this session:**
> Dense alone fails on `BFL-PL-2024-MUMBAI` (product code).  
> Sparse alone fails on `"What if I can't pay this month?"` (semantic).  
> Hybrid catches both. Reranking puts the best answer at rank 1.  
> That's why production RAG systems always use all three layers.
